In [ ]:
import pandas as pd

import great_expectations as gx
import synapseclient

from agoradatatools.gx import GreatExpectationsRunner

context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_nested_field_values_mostly_regex import (
    ExpectColumnNestedObjectRegexRule,
)


# Create Expectation Suite for UI Config Data

## Get Example Data File

In [ ]:
syn = synapseclient.Synapse()
syn.login()

In [ ]:
model_details = syn.get("syn66330545").path

## Create Validator Object on Data File

In [ ]:
df = pd.read_json(model_details)
nested_columns = ["genetic_info", "biomarkers", "pathology"]
df = GreatExpectationsRunner.convert_nested_columns_to_json(df, nested_columns)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "model_details"

## Add Expectations to Validator Object For Each Column

In [ ]:
validator.expect_column_nested_object_regex_rule(column="genetic_info", target_field="ensembl_gene_id", regex_pattern="^ENSMUSG", valid_threshold=0.4)

## Save Expectation Suite

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)

## Create Checkpoint and View Results

## Build Data Docs - Click on Expectation Suite to View All Expectations

In [ ]:
context.build_data_docs()
context.open_data_docs()